# **Hands-on Lab: Interactive Visual Analytics with Folium**


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:

In [1]:
#--- HEADER: LIBRARY IMPORTS FOR FOLIUM INTERACTIVE MAPS
#--- Importing Folium and its plugins for creating interactive maps
#--- with markers, clusters, and mouse position tracking.

# folium is the core library for creating interactive maps with Leaflet.js
import folium

# pandas is used for data manipulation and analysis
import pandas as pd

# MarkerCluster groups nearby markers into clusters for better visualization
# When zoomed out, clusters show the number of markers; zooming in reveals individual markers
from folium.plugins import MarkerCluster

# MousePosition displays the latitude/longitude coordinates of the mouse cursor
# on the map in real-time, useful for exploring spatial data
from folium.plugins import MousePosition

# DivIcon allows custom HTML/CSS icons for markers
# This enables text labels or custom styling on map markers
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


## Task 1: Mark all launch sites on a map

First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [2]:
"""
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)
"""

spacex_df = pd.read_csv('/home/mapa8/Documents/CIDSPC/MFaD/C10_M3_A_spacex_launch_geo.csv')
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [3]:
#--- HEADER: SELECT AND GROUP LAUNCH SITE COORDINATES
#--- Selecting relevant columns for map visualization and grouping
#--- by launch site to get unique site coordinates.

# Select relevant sub-columns: 'Launch Site', 'Lat(Latitude)', 'Long(Longitude)', 'class'
# spacex_df is the DataFrame containing launch data with coordinates and class labels
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

# Group by 'Launch Site' and keep the first occurrence of each site
# This removes duplicate entries for the same launch site
# as_index=False prevents the grouping column from becoming the index
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()

# Keep only the site name, latitude, and longitude columns
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

# Display the unique launch sites with their coordinates
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [4]:
#--- HEADER: CREATE BASE MAP WITH NASA JOHNSON SPACE CENTER AS STARTING POINT
#--- Initializing a Folium map centered at NASA Johnson Space Center
#--- with a zoom level that shows the surrounding area.

# Start location is NASA Johnson Space Center (Houston, TX)
# These coordinates are used as the initial center point of the map
nasa_coordinate = [29.559684888503615, -95.0830971930759]

# Create a Folium map object
# location parameter sets the initial center point of the map
# zoom_start=10 sets the initial zoom level (higher = more zoomed in)
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [5]:
#--- HEADER: ADD NASA JOHNSON SPACE CENTER MARKER AND CIRCLE
#--- Adding a circle and a text label marker at NASA Johnson Space Center
#--- to highlight its location on the map.

# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
# folium.Circle() creates a circular overlay on the map
# radius=1000 sets the circle radius in meters (1 km)
# color='#d35400' sets the circle outline color (orange)
# fill=True fills the circle with color
# .add_child(folium.Popup('NASA Johnson Space Center')) adds a popup label when clicked
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))

# Create a blue circle at NASA Johnson Space Center's coordinate with an icon showing its name
# folium.map.Marker() creates a marker at the specified coordinates
# icon=DivIcon() creates a custom HTML-based icon for the marker
# icon_size=(20,20) sets the size of the icon area
# icon_anchor=(0,0) sets the anchor point of the icon (top-left corner)
# html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC' creates an HTML div with the label text
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )

# Add the circle and marker to the map
site_map.add_child(circle)
site_map.add_child(marker)

# Display the map
site_map

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map

In [6]:
#--- HEADER: ADD MARKERS AND CIRCLES FOR ALL LAUNCH SITES
#--- Iterating through each launch site and adding a circle marker
#--- with a popup label showing the site name.

# List of launch site coordinates
# Each entry contains: [Site Name, Latitude, Longitude]
launch_sites = [
    ['CCAFS LC-40', 28.56230197, -80.57735648],
    ['CCAFS SLC-40', 28.56319718, -80.57682003],
    ['KSC LC-39A', 28.57325457, -80.64689529],
    ['VAFB SLC-4E', 34.63283416, -120.6107455]
]

# Iterate through each launch site and add a circle and marker
for site_name, lat, lon in launch_sites:
    # Create coordinates list for the site
    site_coordinate = [lat, lon]
    
    # Create a circle at the launch site with a popup label
    # radius=1000 sets the circle radius in meters (1 km)
    # color='#000000' sets the circle outline color (black)
    # fill=True fills the circle with color
    # fill_color='#000000' sets the fill color (black)
    # .add_child(folium.Popup(site_name)) adds a popup with the site name when clicked
    circle = folium.Circle(
        site_coordinate, 
        radius=1000, 
        color='#000000', 
        fill=True, 
        fill_color='#000000'
    ).add_child(folium.Popup(site_name))
    
    # Create a marker at the launch site with a text label
    # folium.map.Marker() creates a marker at the specified coordinates
    # icon=DivIcon() creates a custom HTML-based icon with the site name
    # icon_size=(20,20) sets the size of the icon area
    # icon_anchor=(0,0) sets the anchor point of the icon (top-left corner)
    # html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name creates an HTML div with the site name
    marker = folium.map.Marker(
        site_coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )
    
    # Add the circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the updated map with all launch sites
site_map

An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [7]:
#--- HEADER: CREATE MAP WITH CIRCLE MARKERS FOR ALL LAUNCH SITES
#--- Initializing a map centered at NASA Johnson Space Center
#--- and adding circle markers for each launch site with popup labels.

# Initialize the map centered at NASA Johnson Space Center
# zoom_start=5 provides a broader view to see all launch sites
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle object based on its coordinate (Lat, Long) values.
# In addition, add Launch site name as a popup label
for site_name, lat, lon in launch_sites:
    # Create coordinates list for the site
    site_coordinate = [lat, lon]
    
    # Create a circle at the launch site
    # radius=1000 sets the circle radius in meters (1 km)
    # color='#000000' sets the circle outline color (black)
    # fill=True fills the circle with color
    # fill_color='#000000' sets the fill color (black)
    # .add_child(folium.Popup(site_name)) adds the site name as a popup label
    circle = folium.Circle(
        site_coordinate, 
        radius=1000, 
        color="#0A44F4", # NOTE mìo, color cambiado
        fill=True, 
        fill_color="#0A44F4"
    ).add_child(folium.Popup(site_name))
    
    # Add the circle to the map
    site_map.add_child(circle)

# Display the map with all launch site circles
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not

In [8]:
#--- HEADER: DISPLAY LAST 10 ROWS OF THE DATAFRAME
#--- Viewing the final 10 entries in the SpaceX dataset to understand
#--- the most recent launches in the dataset.

spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [9]:
#--- HEADER: INITIALIZE MARKER CLUSTER
#--- Creating a MarkerCluster object to group nearby markers
#--- and improve map readability when many markers are present.

# marker_cluster = MarkerCluster()
# MarkerCluster() creates a cluster object that groups nearby markers
# When zoomed out, clusters show the number of markers in that area
# Zooming in reveals individual markers
# This improves map performance and visual clarity
marker_cluster = MarkerCluster()

*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value

In [10]:
#--- HEADER: CREATE MARKER COLOR COLUMN FOR VISUALIZATION
#--- Creating a new column in spacex_df to assign marker colors
#--- based on the landing success (class value).

# Create a new column 'marker_color' in spacex_df
# If class = 1 (success), marker color is green
# If class = 0 (failure), marker color is red
# Apply this mapping to each row using a lambda function
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Display the first few rows to verify the new column
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [11]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`

In [12]:
#--- HEADER: ADD MARKERS TO MARKER CLUSTER FOR EACH LAUNCH
#--- Iterating through each row in the DataFrame and adding a marker
#--- to the marker cluster with color coding based on landing success.

# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# For each launch result in spacex_df, add a folium.Marker to marker_cluster
for index, record in spacex_df.iterrows():
    # Extract coordinates, site name, and class from the record
    lat = record['Lat']
    lon = record['Long']
    site_name = record['Launch Site']
    marker_color = record['marker_color']
    
    # Create a marker at the launch coordinates
    # icon=folium.Icon(color=marker_color) sets the marker color
    # popup=site_name adds the site name as a popup label
    marker = folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color=marker_color),
        popup=site_name
    )
    
    # Add the marker to the marker cluster
    marker_cluster.add_child(marker)

# Display the map with all markers
site_map

In [13]:
# NOTE mìo, "already done"

"""
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map
"""

"\n# Add marker_cluster to current site_map\nsite_map.add_child(marker_cluster)\n\n# for each row in spacex_df data frame\n# create a Marker object with its coordinate\n# and customize the Marker's icon property to indicate if this launch was successed or failed, \n# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']\nfor index, record in spacex_df.iterrows():\n    # TODO: Create and add a Marker cluster to the site map\n    # marker = folium.Marker(...)\n    marker_cluster.add_child(marker)\n\nsite_map\n"

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.

Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [14]:
#--- HEADER: ADD MOUSE POSITION DISPLAY TO MAP
#--- Adding a control that displays the latitude and longitude of the mouse cursor
#--- when hovering over the map, useful for identifying coordinates.

# Define a formatter function to format the coordinates with 5 decimal places
# L.Util.formatNum(num, 5) rounds the number to 5 decimal places
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

# Create a MousePosition object to display coordinates
# position='topright' places the display in the top-right corner of the map
# separator=' Long: ' separates latitude and longitude in the display
# empty_string='NaN' displays when no coordinates are available
# lng_first=False displays latitude first (Lat: XX, Long: XX)
# num_digits=20 sets the number of digits displayed
# prefix='Lat:' labels the latitude value
# lat_formatter=formatter applies the formatter to latitude
# lng_formatter=formatter applies the formatter to longitude
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

# Add the mouse position control to the map
site_map.add_child(mouse_position)

# Display the map
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [15]:
#--- HEADER: CALCULATE DISTANCE BETWEEN TWO COORDINATES
#--- Defining a function that uses the Haversine formula to calculate
#--- the great-circle distance between two points on Earth.

# Import mathematical functions for distance calculation
# sin, cos, sqrt, atan2 are used in the Haversine formula
# radians converts degrees to radians for trigonometric functions
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    # R = 6373.0 is the mean radius of Earth in kilometers
    R = 6373.0

    # Convert latitude and longitude from degrees to radians
    # Trigonometric functions in Python use radians
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    # Calculate the differences in longitude and latitude
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Haversine formula:
    # a = sin²(Δlat/2) + cos(lat1) * cos(lat2) * sin²(Δlon/2)
    # c = 2 * atan2(√a, √(1-a))
    # distance = R * c
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    # Calculate the distance in kilometers
    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [16]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

In [17]:
#--- HEADER: CALCULATE DISTANCE FROM LAUNCH SITE TO COASTLINE
#--- Using MousePosition to find the closest coastline coordinates
#--- and calculating the distance from the launch site.

# Define launch site coordinates (example: CCAFS SLC-40)
launch_site_lat = 28.56319718
launch_site_lon = -80.57682003

# Define coastline coordinates (found using MousePosition on the map)
# These coordinates represent the closest point on the coastline
coastline_lat = 28.56367
coastline_lon = -80.57163

# Calculate the distance between the launch site and the coastline
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Print the distance
print(f"Distance from launch site to coastline: {distance_coastline:.2f} km")

Distance from launch site to coastline: 0.51 km


In [18]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point

In [19]:
#--- HEADER: ADD POLYLINE BETWEEN LAUNCH SITE AND COASTLINE
#--- Creating a line on the map connecting the launch site to the coastline
#--- to visually represent the distance between them.

# Define the coordinates for the PolyLine
# locations takes a list of [lat, lon] points to connect
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]

# Create a folium.PolyLine object using the coastline coordinates and launch site coordinate
# locations=coordinates sets the points to connect
# weight=1 sets the line thickness (1 is thin, higher values are thicker)
# color='blue' sets the line color for visibility
lines = folium.PolyLine(locations=coordinates, weight=1, color='blue')

# Add the PolyLine to the map
site_map.add_child(lines)

# Display the map with the line
site_map

Your updated map with distance line should look like the following screenshot:

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first

A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [20]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


In [21]:
#--- HEADER: ADD MARKER AND LINE TO CLOSEST CITY
#--- Creating a marker at a nearby city and drawing a line
#--- to visually represent the distance from the launch site.

# Define coordinates for a nearby city (example: Cocoa Beach, FL)
# These coordinates can be found using MousePosition on the map
city_lat = 28.38611
city_lon = -80.60806
city_coordinate = [city_lat, city_lon]

# Calculate the distance from the launch site to the city
# distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

# Create a marker at the city location with distance label
# icon=DivIcon() creates a custom HTML label showing the distance
# icon_size=(20,20) sets the size of the icon area
# icon_anchor=(0,0) sets the anchor point of the icon (top-left corner)
# html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city)
#   - Creates an HTML div with the distance formatted to 2 decimal places
#   - Color is orange (#d35400) for visibility
city_marker = folium.Marker(
    city_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),
    )
)

# Add the city marker to the map
site_map.add_child(city_marker)

# Create a PolyLine between the launch site and the city
# locations takes a list of [lat, lon] points to connect
city_coordinates = [[launch_site_lat, launch_site_lon], [city_lat, city_lon]]
# color='green' sets the line color for visibility
# weight=1 sets the line thickness (1 is thin)
city_line = folium.PolyLine(locations=city_coordinates, weight=1, color='green')

# Add the line to the map
site_map.add_child(city_line)

# Display the map with the city marker and line
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.
